# 02. Modeling

01_eda_and_features.ipynb で作成した特徴量を使い、CatBoost で 5-fold CV 学習を行う。

- 5-fold StratifiedKFold で OOF 予測を取得
- 独立した val セットで最終評価

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 5

## 1. データ読み込み

01 で保存した前処理済みデータを使用。

In [2]:
train = pd.read_csv("../dataset/processed/train_features.csv")
val = pd.read_csv("../dataset/processed/val_features.csv")
test = pd.read_csv("../dataset/processed/test_features.csv")
train_labels = pd.read_csv("../dataset/processed/train_labels.csv").squeeze()
val_labels = pd.read_csv("../dataset/processed/val_labels.csv").squeeze()
cat_feature_indices = pd.read_csv(
    "../dataset/processed/cat_feature_indices.csv", header=None
)[0].tolist()

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")
print(f"cat_feature_indices: {cat_feature_indices}")

Train: (567000, 86), Val: (63000, 86), Test: (270000, 86)
cat_feature_indices: [78, 79, 80, 81, 82, 83, 84, 85]


## 2. CatBoost 5-Fold CV

ハイパーパラメータ:
- iterations: 1000
- learning_rate: 0.03
- depth: 6
- early_stopping_rounds: 50
- cat_features でカテゴリ列を指定（CatBoost のネイティブターゲットエンコーディングが適用される）

In [3]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(train, train_labels))

print(f"StratifiedKFold: {N_SPLITS} folds")
for i, (tr_idx, vl_idx) in enumerate(folds):
    print(f"  Fold {i+1}: train={len(tr_idx)}, val={len(vl_idx)}, "
          f"pos_rate={train_labels.iloc[vl_idx].mean():.4f}")

StratifiedKFold: 5 folds
  Fold 1: train=453600, val=113400, pos_rate=0.4483
  Fold 2: train=453600, val=113400, pos_rate=0.4483
  Fold 3: train=453600, val=113400, pos_rate=0.4483
  Fold 4: train=453600, val=113400, pos_rate=0.4483
  Fold 5: train=453600, val=113400, pos_rate=0.4483


In [6]:
oof_preds = np.zeros(len(train))
val_preds = np.zeros(len(val))
test_preds = np.zeros(len(test))

for fold, (tr_idx, vl_idx) in enumerate(folds):
    X_tr = train.iloc[tr_idx]
    y_tr = train_labels.iloc[tr_idx]
    X_vl = train.iloc[vl_idx]
    y_vl = train_labels.iloc[vl_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0,
        early_stopping_rounds=50,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_vl, y_vl),
        cat_features=cat_feature_indices,
        use_best_model=True,
    )

    oof_preds[vl_idx] = model.predict_proba(X_vl)[:, 1]
    val_preds += model.predict_proba(val)[:, 1] / N_SPLITS
    test_preds += model.predict_proba(test)[:, 1] / N_SPLITS

    fold_auc = roc_auc_score(y_vl, oof_preds[vl_idx])
    print(f"  Fold {fold + 1} AUC: {fold_auc:.6f} "
          f"(best_iteration={model.best_iteration_})")

  Fold 1 AUC: 0.955971 (best_iteration=729)
  Fold 2 AUC: 0.955437 (best_iteration=705)
  Fold 3 AUC: 0.955182 (best_iteration=849)
  Fold 4 AUC: 0.956012 (best_iteration=781)
  Fold 5 AUC: 0.955681 (best_iteration=702)


## 3. 結果

In [7]:
oof_auc = roc_auc_score(train_labels, oof_preds)
val_auc = roc_auc_score(val_labels, val_preds)

print("=" * 40)
print("CatBoost Results")
print("=" * 40)
print(f"OOF AUC: {oof_auc:.6f}")
print(f"Val AUC: {val_auc:.6f}")
print(f"Gap:     {val_auc - oof_auc:+.6f}")

CatBoost Results
OOF AUC: 0.955655
Val AUC: 0.956643
Gap:     +0.000988
